# Modelagem Supervisionada

In [ ]:
#importando bibliotecas
import pandas as pd
import plotly.graph_objects as go
import plotly.figure_factory as ff 

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

pd.set_option('display.max_columns', None)

In [2]:
#importando dados
df = pd.read_pickle('../data/curated/features.pkl')
#df = df.drop(columns=['DESTINATION_AIRPORT_PROFILE', 'ORIGIN_AIRPORT_PROFILE', 'AIRLINE_PROFILE', 'ROUTE_PROFILE'])
df.head()

,MONTH,DAY_OF_WEEK,SCHEDULED_DEPARTURE_HOUR,SCHEDULED_ARRIVAL_HOUR,SCHEDULED_TIME,DISTANCE,ORIGIN_AIRPORT_PROFILE,DESTINATION_AIRPORT_PROFILE,AIRLINE_PROFILE,ROUTE_PROFILE,IS_DELAYED,SEASON_AUTUMN,SEASON_SPRING,SEASON_SUMMER,SEASON_WINTER,TIME_OF_DAY_AFTERNOON,TIME_OF_DAY_EVENING,TIME_OF_DAY_MORNING,TIME_OF_DAY_OVERNIGHT
0,0.253167,-1.456333,1.135593,1.143697,1.624644,1.580078,0.461064,1.202951,-0.848948,-1.500206,1,0,0,0,1,0,1,0,0
1,1.436686,-0.953642,0.716535,0.557100,-1.134818,-1.216751,-1.241973,-0.516591,1.077930,1.222706,1,0,1,0,0,1,0,0,0
2,-1.522113,0.554430,1.135593,0.948165,-1.041952,-1.014511,0.461064,1.202951,1.077930,-1.500206,1,0,0,1,0,0,1,0,0
3,-1.226233,1.559811,-0.121581,-0.029496,0.112246,0.003265,0.461064,1.202951,1.077930,1.222706,0,0,0,1,0,1,0,0,0
4,1.436686,1.559811,-0.331110,-0.029496,0.311246,0.049304,-1.241973,-0.516591,-0.848948,1.222706,0,0,1,0,0,1,0,0,0


Treinamento

In [3]:
X = df.drop(columns=['IS_DELAYED'])
y = df['IS_DELAYED']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Distribuição das classes:\n{y.value_counts(normalize=True)}')
print(f'Treino: {X_train.shape}, Teste: {X_test.shape}')

Distribuição das classes:
IS_DELAYED
1    0.5
0    0.5
Name: proportion, dtype: float64
Treino: (1543796, 18), Teste: (385950, 18)


In [ ]:
def get_best_model(X_train, y_train, X_test, y_test):
    model_configs = {
        'XGBoost': {
            'model': XGBClassifier(tree_method='hist', random_state=42, n_jobs=-1, eval_metric='logloss'),
            'params': {
                'n_estimators': [200, 500],
                'max_depth': [6, 10],
                'learning_rate': [0.01, 0.1],
                'subsample': [0.8, 1.0],
                'colsample_bytree': [0.8, 1.0]
            }
        },
        'RandomForest': {
            'model': RandomForestClassifier(random_state=42, n_jobs=-1),
            'params': {
                'n_estimators': [100, 300],
                'max_depth': [10, 20],
                'min_samples_split': [5, 10],
                'bootstrap': [True, False]
            }
        },
        'GradientBoosting': {
            'model': GradientBoostingClassifier(random_state=42),
            'params': {
                'n_estimators': [100, 200],
                'learning_rate': [0.05, 0.1],
                'max_depth': [3, 5, 8],
                'subsample': [0.8, 1.0]
            }
        }
    }

    results = []
    optimized_models = {}

    print(f"{'Model':<18} | {'Acc CV':<10} | {'Acc Teste':<10} | {'F1 Teste':<10}")
    print("-" * 60)

    for name, config in model_configs.items():
        search = RandomizedSearchCV(
            estimator=config['model'],
            param_distributions=config['params'],
            n_iter=2,
            scoring='accuracy',
            cv=3,
            random_state=42,
            n_jobs=-1
        )
        
        search.fit(X_train, y_train)
        
        best_model = search.best_estimator_
        y_pred = best_model.predict(X_test)
        
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')
        prec = precision_score(y_test, y_pred, average='weighted')
        rec = recall_score(y_test, y_pred, average='weighted')
        
        results.append({
            'Model': name,
            'Acc CV': search.best_score_,
            'Acc Test': acc,
            'F1 Test': f1,
            'Precision Test': prec,
            'Recall Test': rec,
            'Best Params': search.best_params_
        })
        
        optimized_models[name] = best_model
        print(f"{name:<18} | {search.best_score_:<10.4f} | {acc:<10.4f} | {f1:<10.4f}")

    df_results = pd.DataFrame(results).sort_values(by='Acc Test', ascending=False)
    return df_results, optimized_models

In [ ]:
df_results, optimized_model = get_best_model(X_train, y_train, X_test, y_test)

display(df_results)

best_model = optimized_model[df_results.iloc[0]['Model']]

print(f"\nO melhor modelo foi: {df_results.iloc[0]['Model']}")
print(f"Parâmetros: {best_model.iloc[0]['Best Params']}")

Modelo               | Melhor ROC-AUC (CV)  | ROC-AUC (Teste)     
-----------------------------------------------------------------
XGBoost              | 0.6834               | 0.6877              


In [ ]:
def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    x = ['Previsto Pontual', 'Previsto Atrasado']
    y = ['Real Pontual', 'Real Atrasado']
    
    fig = ff.create_annotated_heatmap(
        z=cm, 
        x=x, 
        y=y, 
        annotation_text=cm, 
        colorscale='Blues'
    )
    fig.update_layout(
        title=title,
        xaxis_title='Predição',
        yaxis_title='Realidade',
        template='plotly_white'
    )
    return fig

In [ ]:
fig_xgb = plot_confusion_matrix(y_test, y_pred_xgb, 'Matriz de Confusão: XGBoost')
fig_xgb.show()

In [ ]:
def plot_roc_curve(y_true, y_probs, title):
    fpr, tpr, _ = roc_curve(y_true, y_probs)
    auc_score = roc_auc_score(y_true, y_probs)

    fig = go.Figure()

    fig.add_trace(go.Scatter(
            x=fpr, 
            y=tpr,
            mode='lines',
            name=f'XGBoost (AUC = {auc_score:.4f})',
            line=dict(color='darkblue', width=3)
        )
    )

    fig.add_trace(go.Scatter(
            x=[0, 1], 
            y=[0, 1],
            mode='lines',
            name='Predição Aleatória',
            line=dict(color='red', dash='dash')
        )
    )

    fig.update_layout(
        title=title,
        xaxis_title='Taxa de Falso Positivo (1 - Especificidade)',
        yaxis_title='Taxa de Verdadeiro Positivo (Sensibilidade)',
        height=600,
        template='plotly_white'
    )
    
    fig.show()

    return auc_score

In [ ]:
y_probs_xgb = xgb_model.predict_proba(X_test)[:, 1]
plot_roc_curve(y_test, y_probs_xgb, 'Curva ROC')

0.6737278388699077

Que características aumentam a chance de atraso em um voo?

df_importances = pd.DataFrame({
    'Variável': features,
    'Importância': xgb_model.feature_importances_
}).sort_values(by='Importância', ascending=False)

fig = px.bar(
    df_importances,
    x='Importância',
    y='Variável',
    orientation='h',
    title='Importância das Variáveis Explanatórias'
)
fig.show()